# Train remediation model (runs 12-19)
Supervised, not Isolation Forest -- see PROJECT_STATE_PART2.md for why.

In [ ]:
import sys, json
import pandas as pd
import numpy as np

sys.path.insert(0, "/home/anis/cluster_security/remediation-engine")
from features import FEATURE_COLUMNS

GT_PATH = "/home/anis/cluster_security/ground_truth_log.csv"
LOG_PATH = "/home/anis/cluster_security/remediation-engine/remediation_log.jsonl"
VALID_RUNS = [f"run{i}" for i in range(12, 20)]
print(FEATURE_COLUMNS)

In [ ]:
rows = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        row = {"pod_name": r["pod_name"], "namespace": r["namespace"],
               "timestamp": r["timestamp"], "triggering_rule": r["triggering_rule"]}
        row.update(r["features"])
        rows.append(row)

log_df = pd.DataFrame(rows)
log_df["ts"] = pd.to_datetime(log_df["timestamp"], utc=True)
print(log_df.shape)

In [ ]:
gt_cols = ["run_id","start_utc","end_utc","label","type","pod","expected_rule","detail"]
gt = pd.read_csv(GT_PATH, header=None, names=gt_cols, skiprows=1)
gt = gt.dropna(subset=["run_id"]).copy()
gt = gt[gt["run_id"].isin(VALID_RUNS)].copy()
gt["start_dt"] = pd.to_datetime(gt["start_utc"], format="%Y-%m-%dT%H:%M:%S.%f", utc=True, errors="coerce")
gt["end_dt"]   = pd.to_datetime(gt["end_utc"], format="%Y-%m-%dT%H:%M:%S.%f", utc=True, errors="coerce")
gt = gt.dropna(subset=["start_dt","end_dt"])
print(gt.shape)

In [ ]:
def label_row(pod, ts):
    cand = gt[(gt["pod"] == pod) & (gt["label"].isin(["attack","benign"]))]
    for _, row in cand.iterrows():
        if (row["start_dt"] - pd.Timedelta(seconds=1)) <= ts <= (row["end_dt"] + pd.Timedelta(seconds=3)):
            return row["label"], row["run_id"]
    return None, None

log_df[["label","run_id"]] = log_df.apply(lambda r: pd.Series(label_row(r["pod_name"], r["ts"])), axis=1)
print(log_df["label"].value_counts(dropna=False))

In [ ]:
train_df = log_df.dropna(subset=["label"]).copy()
train_df["target"] = (train_df["label"] == "attack").astype(int)
missing = [c for c in FEATURE_COLUMNS if c not in train_df.columns]
assert not missing, f"Missing feature cols: {missing}"
train_df.to_csv("training_table_v2.csv", index=False)
print(train_df.shape, train_df["target"].value_counts().to_dict())

## Model comparison (GroupKFold by run_id -- never split by row/time)
Dataset is 86% attack (collection artifact, not real-world base rate -- see PROJECT_STATE_PART2.md).
Precision is inflated for any model that defaults toward 'attack'; **recall differentiates
models here more than precision does.** Report both, and caveat precision's meaning in the
writeup: it reflects separability on THIS dataset, not false-positive rate on a quiet cluster.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

X = train_df[FEATURE_COLUMNS].values
y = train_df["target"].values
groups = train_df["run_id"].values
gkf = GroupKFold(n_splits=4)

def eval_model(name, make_clf):
    precisions, recalls, f1s = [], [], []
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        scaler = StandardScaler().fit(X_tr)
        clf = make_clf()
        clf.fit(scaler.transform(X_tr), y_tr)
        pred = clf.predict(scaler.transform(X_te))
        precisions.append(precision_score(y_te, pred, zero_division=0))
        recalls.append(recall_score(y_te, pred, zero_division=0))
        f1s.append(f1_score(y_te, pred, zero_division=0))
    print(f"{name:20s} precision={np.mean(precisions):.3f} recall={np.mean(recalls):.3f} f1={np.mean(f1s):.3f}")
    return np.mean(precisions), np.mean(recalls), np.mean(f1s)

results = {}
results["LogisticRegression"] = eval_model("LogisticRegression", lambda: LogisticRegression(max_iter=1000, class_weight="balanced"))
results["GradientBoosting"]   = eval_model("GradientBoosting",   lambda: GradientBoostingClassifier(random_state=42))
results["RandomForest"]       = eval_model("RandomForest",       lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42))

any_alert = np.ones(len(y))
print(f"{'AnyAlert(baseline)':20s} precision={precision_score(y,any_alert):.3f} recall={recall_score(y,any_alert):.3f}")
crit_idx = FEATURE_COLUMNS.index("max_priority_weight")
crit_pred = (X[:, crit_idx] >= 4).astype(int)
print(f"{'CriticalOnly(baseline)':20s} precision={precision_score(y,crit_pred):.3f} recall={recall_score(y,crit_pred):.3f}")

## Pick best model by recall (at precision >= baseline), then finalize on ALL data
Change `BEST_MODEL` below after seeing Cell above's output if a different model wins.

In [ ]:
BEST_MODEL = "RandomForest"  # <-- set this based on results printed above

model_factories = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "GradientBoosting": lambda: GradientBoostingClassifier(random_state=42),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
}

final_scaler = StandardScaler().fit(X)
X_scaled = final_scaler.transform(X)
final_model = model_factories[BEST_MODEL]()
final_model.fit(X_scaled, y)
print("Trained on full dataset:", X.shape[0], "rows")

if hasattr(final_model, "feature_importances_"):
    imp = sorted(zip(FEATURE_COLUMNS, final_model.feature_importances_), key=lambda t: -t[1])
    for name, v in imp:
        print(f"  {name:35s} {v:.3f}")

## Threshold calibration
Engine needs a probability threshold to decide isolate vs kill. Use predict_proba
on held-out folds (not the final full-data fit) to pick thresholds honestly.

In [ ]:
from sklearn.model_selection import cross_val_predict

cv_scaler = StandardScaler().fit(X)
proba = np.zeros(len(y))
for train_idx, test_idx in gkf.split(X, y, groups):
    s = StandardScaler().fit(X[train_idx])
    clf = model_factories[BEST_MODEL]()
    clf.fit(s.transform(X[train_idx]), y[train_idx])
    proba[test_idx] = clf.predict_proba(s.transform(X[test_idx]))[:, 1]

for thresh in [0.3, 0.5, 0.7, 0.8, 0.9]:
    pred = (proba >= thresh).astype(int)
    print(f"thresh={thresh:.2f} precision={precision_score(y,pred,zero_division=0):.3f} recall={recall_score(y,pred,zero_division=0):.3f}")

In [ ]:
# Set thresholds from the table above:
# WARNING_PROBA -> isolate; CRITICAL_PROBA -> kill (higher confidence needed to kill)
WARNING_PROBA = 0.5   # <-- adjust based on table
CRITICAL_PROBA = 0.8  # <-- adjust based on table
print(f"WARNING_PROBA={WARNING_PROBA} CRITICAL_PROBA={CRITICAL_PROBA}")

## Export model + scaler + config for the engine

In [ ]:
import joblib

joblib.dump(final_model, "/home/anis/cluster_security/remediation-engine/model.pkl")
joblib.dump(final_scaler, "/home/anis/cluster_security/remediation-engine/feature_scaler.pkl")

config = {"feature_columns": FEATURE_COLUMNS, "warning_proba": WARNING_PROBA,
          "critical_proba": CRITICAL_PROBA, "model_type": BEST_MODEL}
with open("/home/anis/cluster_security/remediation-engine/model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved model.pkl, feature_scaler.pkl, model_config.json")